# 00 Project Setup and Public Dataset Download

This notebook initializes the capstone project workflow.

The purpose of this notebook is to confirm that the repository, Python environment, local package imports, public dataset registry, and Kaggle download process are working before any data audit or modeling work begins.

This notebook should answer:

1. Am I running from the correct project root?
2. Is `src/glaucoma_segmentation/` importable?
3. Is the active notebook kernel using the intended `glaucoma-capstone` environment?
4. Can the core project modules import successfully?
5. Which public datasets are registered for use?
6. Which public datasets are selected for download and audit?
7. Are the selected datasets present under the ignored local data directory?

Downloaded public data are stored locally under:

`data/external/kaggle/`

These files are intentionally ignored by Git and should not be committed.

This notebook does **not** create manifests, splits, models, or clinical-data products. Those steps begin in later notebooks.

In [1]:
# 00.01 — Imports

from pathlib import Path
import importlib
import inspect
import os
import platform
import subprocess
import sys
import textwrap
import warnings

from IPython.display import Markdown, display
import pandas as pd
import torch

In [2]:
# 00.02 — Project root and source path setup

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", message="IProgress not found.*")

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

def find_project_root(start_path: Path | None = None) -> Path:
    """
    Walk upward from the current location until the project root is found.

    The project root is identified by the presence of:
    - README.md
    - configs/
    - src/
    """
    start_path = Path(start_path or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "configs").exists()
            and (candidate / "src").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate project root. Expected README.md, configs/, and src/."
    )

PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT, SRC_PATH

(PosixPath('/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy'),
 PosixPath('/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/src'))

In [3]:
# 00.03 — Environment and kernel verification

conda_prefix = os.environ.get("CONDA_PREFIX")
expected_env_python = Path(conda_prefix) / "bin" / "python" if conda_prefix else None

environment_check_df = pd.DataFrame(
    [
        {
            "check": "Project root",
            "value": str(PROJECT_ROOT),
        },
        {
            "check": "Source path",
            "value": str(SRC_PATH),
        },
        {
            "check": "Source package exists",
            "value": (SRC_PATH / "glaucoma_segmentation").exists(),
        },
        {
            "check": "Notebook Python executable",
            "value": sys.executable,
        },
        {
            "check": "Python version",
            "value": sys.version.replace("\n", " "),
        },
        {
            "check": "CONDA_PREFIX",
            "value": conda_prefix or "[not set]",
        },
        {
            "check": "Expected conda env Python",
            "value": str(expected_env_python) if expected_env_python else "[not available]",
        },
        {
            "check": "Platform",
            "value": platform.platform(),
        },
        {
            "check": "PyTorch version",
            "value": torch.__version__,
        },
        {
            "check": "CUDA available",
            "value": torch.cuda.is_available(),
        },
        {
            "check": "CUDA device count",
            "value": torch.cuda.device_count(),
        },
    ]
)

environment_check_df

,check,value
0,Project root,/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy
1,Source path,/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy/src
2,Source package exists,True
3,Notebook Python executable,/home/gsr3qz/.conda/envs/glaucoma-capstone/bin/python
4,Python version,"3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:34:02) [GCC 14.3.0]"
5,CONDA_PREFIX,/home/gsr3qz/.conda/envs/glaucoma-capstone
6,Expected conda env Python,/home/gsr3qz/.conda/envs/glaucoma-capstone/bin/python
7,Platform,Linux-4.18.0-553.124.1.el8_10.x86_64-x86_64-with-glibc2.28
8,PyTorch version,2.4.0
9,CUDA available,True


In [4]:
# 00.04 — Core project module import check

core_project_modules = [
    "glaucoma_segmentation.data.dataset_registry",
    "glaucoma_segmentation.data.kaggle_download",
    "glaucoma_segmentation.data.segmentation_dataset",
    "glaucoma_segmentation.data.dataloaders",
    "glaucoma_segmentation.nets.model_factory",
    "glaucoma_segmentation.nets.losses",
    "glaucoma_segmentation.evaluation.metrics",
]

module_check_rows = []

for module_name in core_project_modules:
    try:
        importlib.import_module(module_name)
        module_check_rows.append(
            {
                "module": module_name,
                "status": "OK",
                "error": "",
            }
        )
    except Exception as exc:
        module_check_rows.append(
            {
                "module": module_name,
                "status": "FAIL",
                "error": f"{type(exc).__name__}: {exc}",
            }
        )

module_check_df = pd.DataFrame(module_check_rows)
module_check_df

,module,status,error
0,glaucoma_segmentation.data.dataset_registry,OK,
1,glaucoma_segmentation.data.kaggle_download,OK,
2,glaucoma_segmentation.data.segmentation_dataset,OK,
3,glaucoma_segmentation.data.dataloaders,OK,
4,glaucoma_segmentation.nets.model_factory,OK,
5,glaucoma_segmentation.nets.losses,OK,
6,glaucoma_segmentation.evaluation.metrics,OK,


## 00.05 — Public dataset registry

The public dataset registry defines which external datasets are known to the project and how they should be downloaded or audited.

At this stage, the goal is not to declare every dataset training-ready. The goal is to confirm which datasets are registered, which are selected for the current public-data workflow, and where they are expected to live locally on Rivanna.

The current project strategy is:

1. Use public labeled fundus datasets first to build and validate the segmentation pipeline.
2. Start with the cleanest available public segmentation data.
3. Extend to additional public datasets only after image-mask pairing and visual QA are confirmed.
4. Bring sponsor/clinical data into the workflow later as a holdout/adaptation target, not as the first development step.

In [5]:
# 00.06 — Project helper imports after source path setup

from glaucoma_segmentation.data.dataset_registry import (
    list_dataset_keys,
    select_datasets,
)

# Import the Kaggle helper module itself rather than assuming a specific
# function name. The next cell inspects which public functions it currently
# exposes.
kaggle_download_module = importlib.import_module(
    "glaucoma_segmentation.data.kaggle_download"
)

print("Dataset registry helpers imported successfully.")
print("Kaggle download module imported successfully.")

Dataset registry helpers imported successfully.
Kaggle download module imported successfully.


In [6]:
# 00.07 — Inspect available Kaggle download helpers

kaggle_helper_rows = []

for name in dir(kaggle_download_module):
    if name.startswith("_"):
        continue

    obj = getattr(kaggle_download_module, name)

    kaggle_helper_rows.append(
        {
            "name": name,
            "type": type(obj).__name__,
            "callable": callable(obj),
        }
    )

kaggle_helpers_df = pd.DataFrame(kaggle_helper_rows).sort_values(
    ["callable", "name"],
    ascending=[False, True],
)

kaggle_helpers_df

,name,type,callable
0,Any,_AnyMeta,True
1,Path,type,True
3,check_kaggle_auth,function,True
4,check_kaggle_cli,function,True
5,download_default_kaggle_datasets,function,True
6,download_kaggle_dataset,function,True
7,download_kaggle_datasets,function,True
8,find_project_root,function,True
9,get_kaggle_executable,function,True
10,resolve_local_dir,function,True


In [7]:
# 00.08 — List registered public datasets

all_dataset_keys = list_dataset_keys()

registered_datasets_df = pd.DataFrame(
    {
        "dataset_key": all_dataset_keys,
    }
)

registered_datasets_df

,dataset_key
0,glaucoma_fundus_imaging_bundle
1,papila
2,refuge2_cross
3,drishti_gs
4,rim_one
5,smdg
6,origa_prior_group


In [8]:
# 00.09 — Select datasets for current public-data workflow

DATASET_KEYS = [
    "glaucoma_fundus_imaging_bundle",
    "papila",
]

selected_datasets = select_datasets(DATASET_KEYS)

selected_dataset_rows = []

status_by_key = {
    "glaucoma_fundus_imaging_bundle": "active_first_working_public_source",
    "papila": "registered_near_term_candidate_pending_path_validation",
}

role_by_key = {
    "glaucoma_fundus_imaging_bundle": "confirm download/local presence before Notebook 01 audit",
    "papila": "confirm download/local presence before Notebook 01/02 path validation",
}

for dataset in selected_datasets:
    if isinstance(dataset, dict):
        dataset_key = dataset.get("key") or dataset.get("dataset_key") or dataset.get("name") or str(dataset)
    else:
        dataset_key = str(dataset)

    selected_dataset_rows.append(
        {
            "selected_dataset_key": dataset_key,
            "current_status": status_by_key.get(dataset_key, "selected_for_public_data_workflow"),
            "notebook_00_role": role_by_key.get(dataset_key, "confirm local presence before downstream audit"),
        }
    )

selected_datasets_df = pd.DataFrame(selected_dataset_rows)
selected_datasets_df

,selected_dataset_key,current_status,notebook_00_role
0,glaucoma_fundus_imaging_bundle,active_first_working_public_source,confirm download/local presence before Notebook 01 audit
1,papila,registered_near_term_candidate_pending_path_validation,confirm download/local presence before Notebook 01/02 path validation


In [9]:
# 00.10 — Inspect Kaggle helper function signatures

kaggle_helper_names = [
    "check_kaggle_cli",
    "check_kaggle_auth",
    "resolve_local_dir",
    "download_kaggle_dataset",
    "download_kaggle_datasets",
    "download_default_kaggle_datasets",
]

kaggle_signature_rows = []

for helper_name in kaggle_helper_names:
    helper = getattr(kaggle_download_module, helper_name, None)

    if helper is None:
        kaggle_signature_rows.append(
            {
                "helper": helper_name,
                "available": False,
                "signature": "",
                "docstring_first_line": "",
            }
        )
        continue

    docstring = inspect.getdoc(helper) or ""
    first_docstring_line = docstring.splitlines()[0] if docstring else ""

    kaggle_signature_rows.append(
        {
            "helper": helper_name,
            "available": True,
            "signature": f"{helper_name}{inspect.signature(helper)}",
            "docstring_first_line": first_docstring_line,
        }
    )

kaggle_signatures_df = pd.DataFrame(kaggle_signature_rows)
kaggle_signatures_df

,helper,available,signature,docstring_first_line
0,check_kaggle_cli,True,check_kaggle_cli() -> 'None',Confirm that the Kaggle CLI is available.
1,check_kaggle_auth,True,"check_kaggle_auth() -> 'dict[str, str]'",Check whether Kaggle authentication is available.
2,resolve_local_dir,True,resolve_local_dir(local_dir: 'str | Path') -> 'Path',Resolve a dataset local_dir from the YAML registry.
3,download_kaggle_dataset,True,"download_kaggle_dataset(dataset_key: 'str', dataset_metadata: 'dict[str, Any]', unzip: 'bool' = True, force: 'bool' = False) -> 'dict[st...",Download one Kaggle dataset using the Kaggle CLI.
4,download_kaggle_datasets,True,"download_kaggle_datasets(dataset_keys: 'list[str]', unzip: 'bool' = True, force: 'bool' = False, registry_path: 'str | Path | None' = No...",Download selected Kaggle datasets by registry key.
5,download_default_kaggle_datasets,True,"download_default_kaggle_datasets(unzip: 'bool' = True, force: 'bool' = False, registry_path: 'str | Path | None' = None) -> 'list[dict[s...",Download all datasets marked download_now: true in the registry.


In [10]:
# 00.11 — Kaggle CLI and authentication check

kaggle_check_rows = []

try:
    kaggle_download_module.check_kaggle_cli()
    kaggle_check_rows.append(
        {
            "check": "Kaggle CLI available",
            "status": "OK",
            "details": "Kaggle command-line interface is available to this notebook kernel.",
        }
    )
except Exception as exc:
    kaggle_check_rows.append(
        {
            "check": "Kaggle CLI available",
            "status": "FAIL",
            "details": f"{type(exc).__name__}: Kaggle CLI check failed.",
        }
    )

try:
    # Do not display returned auth details because successful authentication
    # may include sensitive token-related information.
    _ = kaggle_download_module.check_kaggle_auth()
    kaggle_check_rows.append(
        {
            "check": "Kaggle authentication available",
            "status": "OK",
            "details": "Kaggle authentication is available. Sensitive auth details intentionally not displayed.",
        }
    )
except Exception as exc:
    kaggle_check_rows.append(
        {
            "check": "Kaggle authentication available",
            "status": "FAIL",
            "details": f"{type(exc).__name__}: Kaggle authentication is not currently available in this notebook kernel.",
        }
    )

kaggle_check_df = pd.DataFrame(kaggle_check_rows)
kaggle_check_df

,check,status,details
0,Kaggle CLI available,OK,Kaggle command-line interface is available to this notebook kernel.
1,Kaggle authentication available,FAIL,RuntimeError: Kaggle authentication is not currently available in this notebook kernel.


In [11]:
# 00.12 — Inspect local Kaggle data directories

KAGGLE_DATA_ROOT = PROJECT_ROOT / "data" / "external" / "kaggle"

local_dataset_rows = []

for dataset_key in DATASET_KEYS:
    dataset_dir = KAGGLE_DATA_ROOT / dataset_key

    if dataset_dir.exists():
        top_level_items = sorted(dataset_dir.iterdir())
        top_level_preview = [item.name for item in top_level_items[:10]]

        recursive_file_count = sum(1 for item in dataset_dir.rglob("*") if item.is_file())
        recursive_dir_count = sum(1 for item in dataset_dir.rglob("*") if item.is_dir())

        status = "PRESENT"
    else:
        top_level_preview = []
        recursive_file_count = 0
        recursive_dir_count = 0
        status = "MISSING"

    local_dataset_rows.append(
        {
            "dataset_key": dataset_key,
            "expected_local_dir": str(dataset_dir.relative_to(PROJECT_ROOT)),
            "status": status,
            "recursive_file_count": recursive_file_count,
            "recursive_dir_count": recursive_dir_count,
            "top_level_preview": top_level_preview,
        }
    )

local_dataset_presence_df = pd.DataFrame(local_dataset_rows)
local_dataset_presence_df

,dataset_key,expected_local_dir,status,recursive_file_count,recursive_dir_count,top_level_preview
0,glaucoma_fundus_imaging_bundle,data/external/kaggle/glaucoma_fundus_imaging_bundle,PRESENT,21562,41,"[G1020, ORIGA, REFUGE, models]"
1,papila,data/external/kaggle/papila,PRESENT,2965,10,[PapilaDB-PAPILA-17f8fa7746adb20275b5b6a0d99dc9dfe3007e9f]


In [12]:
# 00.13 — Optional controlled Kaggle download

RUN_KAGGLE_DOWNLOAD = False
FORCE_REDOWNLOAD = False
UNZIP_DOWNLOADS = True

download_decision_rows = []

all_selected_data_present = (
    local_dataset_presence_df["status"].eq("PRESENT").all()
)

kaggle_auth_available = (
    kaggle_check_df
    .query("check == 'Kaggle authentication available'")["status"]
    .eq("OK")
    .any()
)

if RUN_KAGGLE_DOWNLOAD:
    if not kaggle_auth_available:
        raise RuntimeError(
            "RUN_KAGGLE_DOWNLOAD is True, but Kaggle authentication is not available. "
            "Set up Kaggle credentials first or set RUN_KAGGLE_DOWNLOAD = False."
        )

    download_results = kaggle_download_module.download_kaggle_datasets(
        dataset_keys=DATASET_KEYS,
        unzip=UNZIP_DOWNLOADS,
        force=FORCE_REDOWNLOAD,
    )

    download_decision_rows.append(
        {
            "action": "download_attempted",
            "reason": "RUN_KAGGLE_DOWNLOAD was set to True.",
            "selected_datasets": DATASET_KEYS,
            "results_type": type(download_results).__name__,
        }
    )
else:
    download_decision_rows.append(
        {
            "action": "download_skipped",
            "reason": (
                "RUN_KAGGLE_DOWNLOAD is False. "
                "This is expected when selected datasets are already present locally."
            ),
            "selected_datasets": DATASET_KEYS,
            "all_selected_data_present": all_selected_data_present,
        }
    )

download_decision_df = pd.DataFrame(download_decision_rows)
download_decision_df

,action,reason,selected_datasets,all_selected_data_present
0,download_skipped,RUN_KAGGLE_DOWNLOAD is False. This is expected when selected datasets are already present locally.,"[glaucoma_fundus_imaging_bundle, papila]",True


In [13]:
# 00.14 — Notebook 00 handoff summary

module_imports_ok = module_check_df["status"].eq("OK").all()
selected_data_present = local_dataset_presence_df["status"].eq("PRESENT").all()

kaggle_cli_ok = (
    kaggle_check_df
    .query("check == 'Kaggle CLI available'")["status"]
    .eq("OK")
    .any()
)

kaggle_auth_ok = (
    kaggle_check_df
    .query("check == 'Kaggle authentication available'")["status"]
    .eq("OK")
    .any()
)

notebook_00_handoff_df = pd.DataFrame(
    [
        {
            "handoff_item": "Project root located",
            "status": "OK" if PROJECT_ROOT.exists() else "CHECK",
            "notes": str(PROJECT_ROOT),
        },
        {
            "handoff_item": "Source package importable",
            "status": "OK" if module_imports_ok else "CHECK",
            "notes": "Core project modules imported successfully." if module_imports_ok else "One or more module imports failed.",
        },
        {
            "handoff_item": "Notebook kernel environment",
            "status": "OK" if "glaucoma-capstone" in sys.executable else "CHECK",
            "notes": sys.executable,
        },
        {
            "handoff_item": "CUDA availability",
            "status": "OK" if torch.cuda.is_available() else "CPU_ONLY",
            "notes": f"CUDA device count: {torch.cuda.device_count()}",
        },
        {
            "handoff_item": "Kaggle CLI",
            "status": "OK" if kaggle_cli_ok else "CHECK",
            "notes": "Kaggle CLI is available." if kaggle_cli_ok else "Kaggle CLI is not available.",
        },
        {
            "handoff_item": "Kaggle authentication",
            "status": "OK" if kaggle_auth_ok else "NOT_AVAILABLE",
            "notes": (
                "Authentication is available."
                if kaggle_auth_ok
                else "Authentication is not currently available, but selected datasets are already present locally."
            ),
        },
        {
            "handoff_item": "Selected public data folders",
            "status": "OK" if selected_data_present else "CHECK",
            "notes": f"Selected datasets: {DATASET_KEYS}",
        },
        {
            "handoff_item": "Next notebook",
            "status": "READY" if selected_data_present and module_imports_ok else "BLOCKED",
            "notes": "Proceed to 01_data_audit_and_manifest.ipynb.",
        },
    ]
)

notebook_00_handoff_df

,handoff_item,status,notes
0,Project root located,OK,/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy
1,Source package importable,OK,Core project modules imported successfully.
2,Notebook kernel environment,OK,/home/gsr3qz/.conda/envs/glaucoma-capstone/bin/python
3,CUDA availability,OK,CUDA device count: 1
4,Kaggle CLI,OK,Kaggle CLI is available.
5,Kaggle authentication,NOT_AVAILABLE,"Authentication is not currently available, but selected datasets are already present locally."
6,Selected public data folders,OK,"Selected datasets: ['glaucoma_fundus_imaging_bundle', 'papila']"
7,Next notebook,READY,Proceed to 01_data_audit_and_manifest.ipynb.
